# Gemini を使用したセマンティック マッチング

  
  
**ユースケースの説明:** あるオンライン小売企業は、自社の在庫商品の価格と競合他社が提供する価格を比較するためにウェブを調査しています。同社の最優先事項は、2 つの製品ウェブページの情報を比較し、ページ上の異なる製品説明が同一の製品に対応しているかどうか（これを「マッチ（一致）」と呼びます）を示す分類を出力するモデルを実装することです。ここでは、ライプチヒ大学によって作成された [エンティティ解決ベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) 内の [Amazon-Google Products データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。

**モデルの説明:** このノートブックでは、Agent Platform で Gemini API を使用して製品説明をマッチングする方法を説明します。[Amazon-GoogleProducts データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) には、Amazon または Google のウェブサイトからスクレイピングされた製品情報が含まれています。製品のタイトル、説明、価格、メーカーが含まれますが、これら 2 つのウェブサイトでは表現が異なっています。このノートブックでは、製品タイトルのみを使用したモデルの構築に焦点を当てます。考え方はシンプルで、製品タイトルが一致しているかどうかを Gemini API に問い合わせるプロンプトを作成します。製品に関する情報はこれらのタイトルのみに限定されていますが、それでもテストセットで 100% に近い精度を達成できます。

**評価方法:** データセットに対するプロンプト設計の過学習を防ぐため、ペアになった説明のデータセット サンプルを、まず 60 件のサンプルを含む評価セット（evalset）と、同じく 60 件のサンプルからなるテストセット（testset）に分割します。60 件というサンプルの選択は、Agent Platform における Gemini API の現在の制限クォータ（1 分あたり 60 リクエスト）に合わせています。続いて、evalset を使用してプロンプトを考案し、testset でモデルの精度を報告します。評価セットとテストセットはいずれも概ねバランスが取れています。

## セットアップ

In [ ]:
import random

import pandas as pd
from google import genai
from google.genai.types import GenerateContentConfig

pd.options.display.max_colwidth = 1000

## データセットの探索

Google および Amazon のウェブサイトからスクレイピングされた [製品情報のデータセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。このデータセットは、ライプチヒ大学による [セマンティック マッチングおよびエンティティ解決のベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) の一部です。このリポジトリに含まれる次の 3 つのテーブルで構成されています。

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

最初のテーブルには、製品のタイトル、説明、メーカーなど、Amazon に掲載されている製品情報が含まれています。

In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon.head()

2 番目のテーブルには、Google のウェブサイトからスクレイピングされた製品情報に関して同様の内容が含まれています。

In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google.head()

最後のテーブルには、同じ製品に対応する両ウェブサイトの製品情報のマッチングが含まれています（2 つのウェブサイトで表現が異なっている可能性があります）。

In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")
matching.head()

この生データから、あらかじめ評価セットとテストセットを生成してあります。

```python
../data/product_matching_eval.csv
../data/product_matching_test.csv
```


評価セット（eval set）を使用してプロンプトを設計し、テストセット（test set）を使用してそのプロンプトに対する Gemini API のパフォーマンスを評価します。これにより、報告するパフォーマンスは、未知の製品説明のペアに対する実際のパフォーマンスにより近くなります。


評価セットとテストセットの分割を生成するために、下のセルの関数を使用しました。この関数は、`matching` DataFrame 内の一致する製品 ID のサンプル（サイズは `SAMPLE_SIZE` で指定）を取得し、`google` および `amazon` DataFrame に含まれる Google と Amazon の製品情報を結合します。次に、一致する Google と Amazon の説明のペアを抽出し、`description_1`（Google）、`description_2`（Amazon）、およびターゲット列 `match`（現時点では一致するペアのみのため値は `yes`）を含む一致説明テーブルを作成します。
一致しない製品の説明ペアを作成するために、最初の説明を固定したまま 2 番目の説明列を並べ替え（置換）、この一致しない説明の新しい DataFrame を一致する説明の DataFrame に連結します。結果のテーブルをシャッフルし、2 つの同じサイズの DataFrame に分割して、評価セットおよびテストセットの分割としてディスクに保存します。

製品の説明として `title` 列のみを使用していることに注目してください。したがって、生データにはもっと多くの情報が存在します。それにもかかわらず、Gemini がテストセットで 100% に近いパフォーマンスを達成することを確認できます。これは驚くべきことです！

**注:** データの別のサンプルで評価セットとテストセットを再生成したい場合は、最後の行のコメントを解除してください。

In [ ]:
SAMPLE_SIZE = 60


def generate_test_and_eval_sets(sample_size=SAMPLE_SIZE):
    sample = matching.sample(sample_size)

    # 一致する ID の DataFrame に製品情報を結合
    matched_products = sample.merge(
        right=amazon, how="left", on="idAmazon"
    ).merge(right=google, how="left", on="idGoogleBase")

    google_descriptions = list(matched_products["google_title"])
    amazon_descriptions = list(matched_products["amazon_title"])

    # 一致する説明の DataFrame を作成
    matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions,
            "match": "yes",
        }
    )

    # 一致しない説明の DataFrame を作成
    amazon_descriptions_perm = [
        amazon_descriptions[i - 1] for i in range(len(amazon_descriptions))
    ]
    not_matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions_perm,
            "match": "no",
        }
    )

    full_dataset = pd.concat(
        [matching_descriptions, not_matching_descriptions], axis=0
    ).sample(len(matching_descriptions) * 2)

    evalset = full_dataset[:sample_size]
    testset = full_dataset[sample_size : 2 * sample_size]
    evalset.to_csv("../data/product_matching_eval.csv", index=None)
    testset.to_csv("../data/product_matching_test.csv", index=None)


# 別のデータサンプルを生成する場合はコメントを解除
# generate_test_and_eval_sets()

次のセルでは、事前に生成した評価用およびテスト用のデータセットを読み込みます。2 つの CSV ファイルには、製品説明ペアの例が 60 件ずつ含まれています。各ペアには、説明が同一の製品を表している場合は `yes`、そうでない場合は `no` の `match` 値がラベル付けされています。`description_1` は Google の製品タイトルから、`description_2` は Amazon の製品から取得されています。

In [ ]:
evalset = pd.read_csv("../data/product_matching_eval.csv")
testset = pd.read_csv("../data/product_matching_test.csv")

このデータセットのいくつかのエントリを簡単に確認してみましょう。

In [ ]:
evalset.head()

どちらの分割も概ねバランスが取れています。両方の分割が概ね均等であることを確認するために、各分割のクラス インスタンス数をカウントします。

In [ ]:
evalset.match.value_counts()

In [ ]:
testset.match.value_counts()

# モデルの実装

まず、Gen AI SDK を使用してクライアントをインスタンス化します。Google が開発した大規模言語モデル（LLM）である Gemini の `gemini-3.8-flash` バージョンを使用します。

In [ ]:
MODEL = "gemini-3.8-flash"

client = genai.Client(vertexai=True, location="global")

このクライアントを使用して、次のセルで 2 つの製品説明とパラメータ化されたプロンプトを入力として受け取り、Gemini モデルが製品説明が一致していると判断したかどうかに応じて `yes` または `no` を出力するシンプルな関数を実装します。

In [ ]:
def are_products_matching(d1, d2, prompt):
    prompt = prompt.format(desc1=d1, desc2=d2)
    generation_config = GenerateContentConfig(
        temperature=0.1,
        top_p=0.8,
        top_k=40,
    )
    answer = client.models.generate_content(
        model=MODEL, contents=prompt, config=generation_config
    )
    return answer.text.strip()

次のセルでは、指定したプロンプトがこのユースケースで機能しているかどうかを評価セットで素早くテストできます。

In [ ]:
PROMPT = """
Are the following two descriptions describing the same product?

EXAMPLE:

Description 1: {desc1}
Description 2: {desc2}

If they are similar, reply with "yes" and if they are not similar, reply with "no".
Answer only with "yes" or "no"
"""

index = random.randint(0, len(evalset) - 1)

d1 = evalset.iloc[index]["description_1"]
d2 = evalset.iloc[index]["description_2"]
ground_truth = evalset.iloc[index]["match"]
prediction = are_products_matching(d1, d2, prompt=PROMPT)


print(
    f"""
Are the following two descriptions describing the same product?

Description 1: {d1}

Descriptions 2: {d2}

MODEL ANSWER: {prediction}
GROUND TRUTH: {ground_truth}
"""
)

# モデルの分析

次に、テストセットにおけるモデルのパフォーマンスを分析します。

大規模言語モデルは、プロンプトで丁寧に指示した場合でも、"yes" や "no" 以外のものを出力することがあります。これは、セーフティ フィルターがトリガーされたり、モデルが単に質問を理解できなかったりすることが原因として考えられます。したがって、まずモデルが回答に失敗したリクエストの割合を特定する必要があります。このケースでは約 6% であり、許容範囲内と言えます。プロンプト エンジニアリングやモデル チューニングをさらに進めることで、この数値を低減できる可能性があります。

考慮すべき 2 つ目の側面は、成功したリクエスト（つまり、出力が実際に `yes` または `no` であったリクエスト）に対するモデルのパフォーマンスです。
テストセットのバランスが取れているため、モデルの正解率（accuracy）を計算することができ、その値は 98% です。これは、テストセット内で正解と異なる予測を受け取った例が 1 件のみであったことを意味します。

## テストセットのスコアリング

評価を簡素化するために、次のセルで `testset` に `prediction` 列を追加する関数を実装します。この列には、Gemini API から受け取った予測が含まれます。

In [ ]:
def apply_prompt(prompt, dataset):
    scored_dataset = dataset.copy()
    scored_dataset["predictions"] = scored_dataset.apply(
        lambda row: are_products_matching(
            row.description_1, row.description_2, prompt=PROMPT
        ),
        axis=1,
    )
    return scored_dataset

`evalset` で設計したシンプルなプロンプトを使用して、この関数を `testset` に適用してみましょう。

In [ ]:
scored_testset = apply_prompt(prompt=PROMPT, dataset=testset)

Gemini API へのリクエストは 1 分あたり 60 リクエストに制限されているため、必要に応じてオフラインで分析できるようにスコアリング結果をディスクに保存します。

In [ ]:
scored_testset.to_csv("scored_evalset.csv", index=False)

以下がモデルの予測結果です。一部のリクエストが失敗して空の予測となったものの、ほとんどの例が正しく分類されていることがわかります。これらのケースを個別に分析し、成功したリクエストに対してのみ正解率を計算する必要があります。

In [ ]:
scored_testset.head(10)

### 失敗した予測

下のセルでは、失敗した予測（つまり、`yes` または `no` 以外の予測）の件数をリストします。このような動作にはいくつかの考えられる原因があります。すべての LLM と同様に、Gemini は一連の単語から次に来る可能性が最も高い単語を予測するようにトレーニングされています。そのため、プロンプトで `yes` または `no` で答えるよう明示的に指示した場合でも、特定の製品説明によって Gemini が混乱し、`yes` や `no` 以外の結果になることがあります。もう 1 つの問題は、製品説明内の表現がセーフティ フィルターをトリガーし、Gemini の生の回答が標準の警告テキストに置き換えられる可能性があることです。これらのセーフティ フィルターは、例えば健康や医療問題、暴力などを強く連想させる製品説明内の単語や、その他多くの [セーフティ設定](https://developers.generativeai.google/guide/safety_setting) によってトリガーされることがあります。

In [ ]:
failed_predictions_mask = ~(
    scored_testset.predictions.str.match("yes")
    | scored_testset.predictions.str.match("no")
)

failed_predictions_mask.value_counts()

In [ ]:
proportion_of_failed_predictions = failed_predictions_mask.sum() / len(
    failed_predictions_mask
)
print(
    "Proportion of failed requests:",
    round(proportion_of_failed_predictions, 3) * 100,
    "%",
)

次のセルでは、失敗したリクエストを調査します。一部の用語がセーフティ フィルターをトリガーした可能性がありますが、これにはさらなる調査が必要です。

In [ ]:
scored_testset[failed_predictions_mask]

## 成功した予測に対する正解率（Accuracy）

成功したリクエストに対するモデルの正解率を計算してみましょう。まず、失敗したすべてのリクエストをテストセットから除外します。

In [ ]:
scored_testset_with_predictions = scored_testset[~failed_predictions_mask]

次に、正解の回答数を計算します。

In [ ]:
is_prediction_correct = (
    scored_testset_with_predictions.match
    == scored_testset_with_predictions.predictions
)
is_prediction_correct.value_counts()

モデルの正解率は、正解数を成功した予測数で割った値であり、以下のように計算されます。

In [ ]:
model_accuracy = is_prediction_correct.sum() / len(is_prediction_correct)
print("Model accuracy:", round(model_accuracy, 3))

実際、この小規模なテストセットにおいて、モデルが犯したエラーの件数は次のようになります。

In [ ]:
uncorrect_predictions_mask = (
    scored_testset_with_predictions.match
    != scored_testset_with_predictions.predictions
)
sum(uncorrect_predictions_mask)

エラーの内容を確認してみましょう。

In [ ]:
scored_testset_with_predictions[uncorrect_predictions_mask]

Copyright 2023 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License